# 08 Deep Learning — Evaluation & Comparison with Phase 1

---

## Purpose

Evaluate the two trained DL models on the **same 20 % hold-out test set** used in
Phase 1, then build a comprehensive comparison across all four models.

### Outputs
- Confusion matrices and ROC curves
- Classification reports
- **Phase 1 (ML) vs Phase 2 (DL)** comparison table and bar chart
- Error analysis: disagreement patterns, accuracy by text length, confidence distributions
- Saved metrics (JSON/CSV) and predictions for Phase 3 hybrid model

### Prerequisites
Run `06_dl_data_preprocessing.ipynb` → `07_dl_model_training.ipynb` first.

## 1. Imports and Load Artifacts

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, auc,
)
from IPython.display import Markdown, display

warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.deep_learning import (
    LSTMConfig, TransformerConfig,
    TextClassificationDataset, TransformerDataset,
    BiLSTMClassifier, DistilBERTClassifier,
    evaluate_lstm, evaluate_transformer,
    get_device,
)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 5)

RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
METRICS_DIR = PROJECT_ROOT / 'metrics'
for d in [RESULTS_DIR, FIGURES_DIR, METRICS_DIR]:
    d.mkdir(exist_ok=True)

device = get_device()
print(f'PyTorch {torch.__version__} | Device: {device}')

In [ ]:
# Load test data
test_ids = torch.load(RESULTS_DIR / 'dl_test_ids.pt', weights_only=True)
y_test   = np.load(RESULTS_DIR / 'dl_y_test.npy')

bert_test_enc = torch.load(RESULTS_DIR / 'dl_bert_test.pt', weights_only=True)

X_test_text = np.load(RESULTS_DIR / 'dl_test_texts.npy', allow_pickle=True)

with open(RESULTS_DIR / 'dl_tokenizer.json') as f:
    tok_data = json.load(f)
vocab_size = len(tok_data['word2idx'])

print(f'Test set: {len(y_test):,} samples')
print(f'Word-level tensor: {test_ids.shape}')
print(f'BERT tensor: {bert_test_enc["input_ids"].shape}')

## 2. Load Trained Models

In [ ]:
# Reconstruct BiLSTM and load best checkpoint
lstm_cfg = LSTMConfig(
    vocab_size=vocab_size, embed_dim=128, hidden_dim=256,
    num_layers=2, dropout=0.3, bidirectional=True, num_classes=2,
)
lstm_model = BiLSTMClassifier(lstm_cfg, pad_idx=0)
lstm_model.load_state_dict(torch.load(RESULTS_DIR / 'model_lstm_best.pt', weights_only=True))
lstm_model = lstm_model.to(device)
lstm_model.eval()
print(f'BiLSTM loaded ({sum(p.numel() for p in lstm_model.parameters()):,} params)')

# Reconstruct DistilBERT and load best checkpoint
bert_cfg = TransformerConfig(
    model_name='distilbert-base-uncased', max_length=512,
    dropout=0.3, num_classes=2, freeze_layers=2,
)
bert_model = DistilBERTClassifier(bert_cfg)
bert_model.load_state_dict(torch.load(RESULTS_DIR / 'model_distilbert_best.pt', weights_only=True))
bert_model = bert_model.to(device)
bert_model.eval()
print(f'DistilBERT loaded ({sum(p.numel() for p in bert_model.parameters()):,} params)')

In [ ]:
# Create test DataLoaders
test_dataset = TextClassificationDataset(test_ids, y_test)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

bert_test_ds = TransformerDataset(bert_test_enc, y_test)
bert_test_loader = DataLoader(bert_test_ds, batch_size=16, shuffle=False, num_workers=0)

print(f'Test loaders: LSTM={len(test_loader)} batches, BERT={len(bert_test_loader)} batches')

## 3. Generate Predictions

In [ ]:
criterion = nn.CrossEntropyLoss()

# BiLSTM evaluation
lstm_test_loss, lstm_test_acc, lstm_preds, lstm_probs = evaluate_lstm(
    lstm_model, test_loader, criterion, device,
)

lstm_metrics = {
    'Accuracy':  accuracy_score(y_test, lstm_preds),
    'Precision': precision_score(y_test, lstm_preds),
    'Recall':    recall_score(y_test, lstm_preds),
    'F1':        f1_score(y_test, lstm_preds),
    'ROC-AUC':   roc_auc_score(y_test, lstm_probs),
    'AP':        average_precision_score(y_test, lstm_probs),
}

print('BiLSTM + Attention — Test Metrics')
for k, v in lstm_metrics.items():
    print(f'  {k:12s}: {v:.4f}')

In [ ]:
# DistilBERT evaluation
bert_test_loss, bert_test_acc, bert_preds, bert_probs = evaluate_transformer(
    bert_model, bert_test_loader, criterion, device,
)

bert_metrics = {
    'Accuracy':  accuracy_score(y_test, bert_preds),
    'Precision': precision_score(y_test, bert_preds),
    'Recall':    recall_score(y_test, bert_preds),
    'F1':        f1_score(y_test, bert_preds),
    'ROC-AUC':   roc_auc_score(y_test, bert_probs),
    'AP':        average_precision_score(y_test, bert_probs),
}

print('DistilBERT — Test Metrics')
for k, v in bert_metrics.items():
    print(f'  {k:12s}: {v:.4f}')

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_names = ['Lost', 'Won']

for ax, preds, title in [
    (axes[0], lstm_preds, 'BiLSTM + Attention'),
    (axes[1], bert_preds, 'DistilBERT'),
]:
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{title} — Confusion Matrix')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/dl_confusion_matrices.png')

## 5. Classification Reports

In [ ]:
print('='*60)
print('BiLSTM + Attention — Classification Report')
print('='*60)
print(classification_report(y_test, lstm_preds, target_names=class_names))

print('='*60)
print('DistilBERT — Classification Report')
print('='*60)
print(classification_report(y_test, bert_preds, target_names=class_names))

## 6. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for probs, name, color in [
    (lstm_probs, 'BiLSTM + Attention', '#1f77b4'),
    (bert_probs, 'DistilBERT', '#ff7f0e'),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{name} (AUC = {roc_auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Deep Learning Models', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dl_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/dl_roc_curves.png')

## 7. Phase 1 (ML) vs Phase 2 (DL) — Direct Comparison

Phase 1 models (Logistic Regression and Random Forest) used **engineered tabular
features + precomputed embeddings → PCA (100 components)**.  The DL models here
use **raw conversation text** end-to-end.  Same test set, same stratification.

In [ ]:
# Phase 1 ML results (from metrics/evaluation_metrics.json)
phase1_metrics = {
    'Logistic Regression (ML)': {
        'Accuracy': 0.9634, 'Precision': 0.9504, 'Recall': 0.9777,
        'F1': 0.9639, 'ROC-AUC': 0.9957, 'AP': 0.9957,
    },
    'Random Forest (ML)': {
        'Accuracy': 0.9594, 'Precision': 0.9453, 'Recall': 0.9751,
        'F1': 0.9600, 'ROC-AUC': 0.9951, 'AP': 0.9951,
    },
}

all_results = {
    **phase1_metrics,
    'BiLSTM + Attention (DL)': lstm_metrics,
    'DistilBERT (DL)': bert_metrics,
}

comparison_df = pd.DataFrame(all_results).T
comparison_df = comparison_df[['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'AP']]

print('\n' + '='*80)
print('FULL MODEL COMPARISON — Phase 1 (ML) vs Phase 2 (DL)')
print('='*80)
display(comparison_df.round(4).style.highlight_max(axis=0, color='lightgreen'))

In [ ]:
# Grouped bar chart
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
model_names = list(all_results.keys())
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

x = np.arange(len(metrics_to_plot))
width = 0.18

fig, ax = plt.subplots(figsize=(14, 6))
for i, (model_name, color) in enumerate(zip(model_names, colors)):
    values = [all_results[model_name][m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, values, width, label=model_name, color=color, alpha=0.85)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Phase 1 (ML) vs Phase 2 (DL) — Model Comparison', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim(0.85, 1.02)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ml_vs_dl_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/ml_vs_dl_comparison.png')